# Lab | Aggregating and Filtering Data

**Dataset:** Customer insurance data (continuación de los labs de Data Cleaning/Formatting y Combining/Structuring)

En este laboratorio practicamos:
- Agrupar datos con `groupby()`
- Aplicar funciones de agregación: `count`, `mean`, `median`, `sum`, `min`, `max`, `std`
- Agrupar por múltiples variables
- Filtrar datos antes y después de agregarlos
- Extraer conclusiones a partir de los resultados

## Importar librerías y cargar los datos

In [1]:
import pandas as pd
import numpy as np

print('Librerías importadas correctamente')

Librerías importadas correctamente


In [2]:
# Cargamos y combinamos los tres ficheros igual que en el lab anterior
url1 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv'
url2 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv'
url3 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv'

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)

# Limpiamos nombres de columnas
for d in [df1, df2, df3]:
    d.columns = d.columns.str.lower().str.replace(' ', '_')

# Renombramos 'st' a 'state'
df1.rename(columns={'st': 'state'}, inplace=True)
df2.rename(columns={'st': 'state'}, inplace=True)

# Alineamos columnas y concatenamos
df2 = df2[df1.columns]
df3 = df3[df1.columns]
df = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

# Limpiamos nulos y duplicados
df.dropna(how='all', inplace=True)
df.drop_duplicates(inplace=True)

# Limpiamos customer_lifetime_value (quitamos '%' y convertimos a float)
df['customer_lifetime_value'] = (
    df['customer_lifetime_value']
    .astype(str)
    .str.replace('%', '', regex=False)
    .astype(float)
)

print(f'Shape final: {df.shape}')
df.head()

Shape final: (9134, 11)


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


In [3]:
# Revisamos las columnas disponibles
print(df.columns.tolist())
df.describe()

['customer', 'state', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'vehicle_class', 'total_claim_amount']


,customer_lifetime_value,income,monthly_premium_auto,total_claim_amount
count,9.127000e+03,9134.000000,9134.000000,9134.000000
mean,1.820715e+05,37824.847055,110.393146,430.480412
std,4.410465e+05,30359.232933,581.471461,289.617985
min,1.898008e+03,0.000000,61.000000,0.099007
25%,4.650064e+03,0.000000,68.000000,266.961153
50%,7.714624e+03,34240.000000,83.000000,377.505619
75%,2.619741e+04,62446.500000,109.000000,546.050821
max,5.816655e+06,99981.000000,35354.000000,2893.239678


## Challenge 1 — Clientes con alto importe de reclamación y bajo CLV

Queremos identificar clientes que generan **muchas reclamaciones** (alto `total_claim_amount`)
pero aportan **poco valor a largo plazo** (bajo `customer_lifetime_value`).

Definimos:
- Alto importe de reclamación: por encima del percentil 75
- Bajo CLV: por debajo del percentil 25

In [4]:
# Calculamos los percentiles de referencia
p75_claim = df['total_claim_amount'].quantile(0.75)
p25_clv   = df['customer_lifetime_value'].quantile(0.25)

print(f'Percentil 75 de total_claim_amount: {p75_claim:.2f}')
print(f'Percentil 25 de customer_lifetime_value: {p25_clv:.2f}')

Percentil 75 de total_claim_amount: 546.05
Percentil 25 de customer_lifetime_value: 4650.06


In [5]:
# Filtramos clientes que cumplen ambas condiciones
clientes_riesgo = df[
    (df['total_claim_amount'] > p75_claim) &
    (df['customer_lifetime_value'] < p25_clv)
]

print(f'Número de clientes identificados: {len(clientes_riesgo)}')
clientes_riesgo[['customer', 'state', 'total_claim_amount', 'customer_lifetime_value']].head(10)

Número de clientes identificados: 399


,customer,state,total_claim_amount,customer_lifetime_value
5007,KX23516,California,699.615192,4455.843406
5018,AL78075,Oregon,777.600000,3810.238281
5031,LZ89773,California,624.711448,2326.864833
5045,EY89842,Arizona,820.800000,4216.195220
5132,JS38272,Arizona,725.870693,4270.034394
5142,RN88075,Washington,700.725788,3021.938522
5150,CI68869,Arizona,702.020904,1898.683686
5160,GB66110,California,722.432062,4422.409799
5200,FN44127,Oregon,1176.278800,3508.569533
5203,PG43749,Oregon,639.769748,4318.433036


In [6]:
# Estadísticas descriptivas de este grupo
clientes_riesgo[['total_claim_amount', 'customer_lifetime_value', 'income', 'monthly_premium_auto']].describe()

,total_claim_amount,customer_lifetime_value,income,monthly_premium_auto
count,399.000000,399.000000,399.000000,399.000000
mean,726.677146,3693.501722,16167.223058,98.776942
std,177.003309,689.768662,24668.986331,18.532407
min,546.052144,1898.683686,0.000000,61.000000
25%,590.400000,3235.283618,0.000000,86.000000
50%,681.505066,3834.210713,0.000000,102.000000
75%,788.400000,4265.185062,25927.000000,114.000000
max,1516.383753,4648.539191,97266.000000,135.000000


**Conclusión:** Estos clientes representan un grupo de riesgo para la aseguradora —
generan reclamaciones por encima de la media pero su valor de vida estimado es bajo.
Identificarlos permite revisar sus condiciones de póliza o investigar patrones de fraude.

## Challenge 2 — Número de pólizas por tipo y por género

Agrupamos por `policy_type` y `gender` para ver cómo se distribuyen los clientes
entre las distintas combinaciones de tipo de póliza y género.

In [7]:
# Agrupamos por tipo de póliza y género, contamos clientes
df.groupby(['policy_type', 'gender'])['customer'].count()

policy_type     gender
Corporate Auto  F          978
                Femal        5
                M          945
                Male         7
                female       6
Personal Auto   F         3385
                Femal       10
                M         3249
                Male        32
                female      22
Special Auto    F          194
                Femal        2
                M          174
                Male         1
                female       2
Name: customer, dtype: int64

In [8]:
# Lo mismo pero como tabla más legible con unstack
df.groupby(['policy_type', 'gender'])['customer'].count().unstack('gender')

gender,F,Femal,M,Male,female
policy_type,,,,,
Corporate Auto,978,5,945,7,6
Personal Auto,3385,10,3249,32,22
Special Auto,194,2,174,1,2


**Conclusión:** Podemos ver si algún tipo de póliza tiene una distribución de género
muy desigual respecto a los demás — esto podría indicar un segmento de mercado
específico al que dirigir campañas de marketing.

## Challenge 3 — Ingresos medios por nivel educativo, filtrando clientes con bajo CLV

Calculamos el ingreso medio (`income`) agrupado por nivel educativo,
pero solo para los clientes cuyo `customer_lifetime_value` está por debajo de la mediana.

In [9]:
# Calculamos la mediana del CLV
mediana_clv = df['customer_lifetime_value'].median()
print(f'Mediana de customer_lifetime_value: {mediana_clv:.2f}')

# Filtramos clientes por debajo de la mediana
df_bajo_clv = df[df['customer_lifetime_value'] < mediana_clv]
print(f'Clientes con CLV bajo: {len(df_bajo_clv)} de {len(df)}')

Mediana de customer_lifetime_value: 7714.62
Clientes con CLV bajo: 4563 de 9134


In [10]:
# Ingreso medio por nivel educativo, solo para clientes con CLV bajo
df_bajo_clv.groupby('education')['income'].agg(['count', 'mean', 'median', 'std']).round(2)

,count,mean,median,std
education,,,,
Bachelor,1422,34955.67,29667.5,30476.11
College,1340,36892.16,32270.0,29738.90
Doctor,175,43501.22,39786.0,28279.04
High School or Below,1274,34862.45,28907.5,31597.78
Master,352,44349.65,42544.5,27272.64


**Conclusión:** Entre los clientes de bajo valor de vida, podemos ver si el nivel
educativo está relacionado con el ingreso medio — si los niveles educativos más altos
tienen ingresos similares a los más bajos dentro de este grupo, sugiere que el CLV bajo
no está determinado únicamente por el nivel de ingresos.

## Challenge 4 — Estadísticas de reclamaciones por estado y tipo de póliza

Agrupamos por `state` y `policy_type` para obtener varias métricas del
`total_claim_amount` a la vez usando `agg()`.

In [11]:
# Múltiples agregaciones a la vez con agg()
stats_estado_poliza = df.groupby(['state', 'policy_type'])['total_claim_amount'].agg(
    ['count', 'mean', 'median', 'min', 'max']
).round(2)

stats_estado_poliza

count    mean  median     min      max
state      policy_type                                           
AZ         Corporate Auto     15  501.71  496.80   82.73  1129.93
           Personal Auto      54  419.35  372.00    5.57   980.17
           Special Auto        5  431.34  392.60  336.00   620.01
Arizona    Corporate Auto    318  417.39  370.46    2.57  1454.40
           Personal Auto    1239  425.04  374.72    0.38  2893.24
           Special Auto       72  454.29  355.20   37.96  2294.63
Cali       Corporate Auto     27  381.17  402.07    7.35  1185.99
           Personal Auto      88  371.23  333.85    4.11  1360.80
           Special Auto        5  260.60  307.96   61.77   537.77
California Corporate Auto    712  431.17  371.10    3.25  2452.89
           Personal Auto    2210  443.28  392.79    0.42  2327.17
           Special Auto      108  446.54  422.85   33.89  1310.40
Nevada     Corporate Auto    195  445.59  407.44    0.10  2759.79
           Personal Auto     652  437.22  384.00    4.24  2552.34
           Special Auto       35  429.21  403.20   42.95  1336.93
Oregon     Corporate Auto    540  434.09  392.92    2.17  2345.41
           Personal Auto    1938  431.40  379.20    0.52  2042.57
           Special Auto      123  453.60  451.20    8.08  1398.02
WA         Corporate Auto      3  606.88  639.46  383.17   798.00
           Personal Auto      25  410.17  403.20    4.24   844.48
           Special Auto        2  104.72  104.72   49.80   159.64
Washington Corporate Auto    155  383.33  346.99   32.01  1245.60
           Personal Auto     584  399.32  336.07    2.70  2145.60
           Special Auto       29  356.50  321.60    8.32   759.29

In [12]:
# Filtramos solo las combinaciones estado/póliza con más de 50 clientes
stats_filtrado = stats_estado_poliza[stats_estado_poliza['count'] > 50]
stats_filtrado

count    mean  median    min      max
state      policy_type                                          
AZ         Personal Auto      54  419.35  372.00   5.57   980.17
Arizona    Corporate Auto    318  417.39  370.46   2.57  1454.40
           Personal Auto    1239  425.04  374.72   0.38  2893.24
           Special Auto       72  454.29  355.20  37.96  2294.63
Cali       Personal Auto      88  371.23  333.85   4.11  1360.80
California Corporate Auto    712  431.17  371.10   3.25  2452.89
           Personal Auto    2210  443.28  392.79   0.42  2327.17
           Special Auto      108  446.54  422.85  33.89  1310.40
Nevada     Corporate Auto    195  445.59  407.44   0.10  2759.79
           Personal Auto     652  437.22  384.00   4.24  2552.34
Oregon     Corporate Auto    540  434.09  392.92   2.17  2345.41
           Personal Auto    1938  431.40  379.20   0.52  2042.57
           Special Auto      123  453.60  451.20   8.08  1398.02
Washington Corporate Auto    155  383.33  346.99  32.01  1245.60
           Personal Auto     584  399.32  336.07   2.70  2145.60

**Conclusión:** Filtrar por un mínimo de clientes evita que combinaciones con muy
pocos casos distorsionen la interpretación — una media calculada sobre 2 o 3 clientes
no es representativa, mientras que con más de 50 sí podemos extraer conclusiones fiables
sobre el comportamiento de cada estado y tipo de póliza.

## Challenge 5 — Categorías de ingreso y tasa de reclamación media

Creamos una nueva columna `income_group` agrupando los ingresos en categorías,
siguiendo el mismo enfoque que en clase con `apply()` y una función personalizada.

In [13]:
def create_income_group(income):
    """
    Clasifica el ingreso del cliente en categorías.
    """
    if income == 0:
        return 'Sin ingresos'
    elif income <= 30000:
        return 'Bajo'
    elif income <= 70000:
        return 'Medio'
    else:
        return 'Alto'

# Aplicamos la función para crear la nueva columna
df['income_group'] = df['income'].apply(create_income_group)

df[['income', 'income_group']].head()

,income,income_group
0,0.0,Sin ingresos
1,0.0,Sin ingresos
2,48767.0,Medio
3,0.0,Sin ingresos
4,36357.0,Medio


In [14]:
# Número de clientes por grupo de ingreso
df['income_group'].value_counts()

income_group
Medio           3208
Sin ingresos    2294
Bajo            1900
Alto            1732
Name: count, dtype: int64

In [15]:
# Importe medio de reclamación por grupo de ingreso
df.groupby('income_group')['total_claim_amount'].agg(['count', 'mean']).round(2)

,count,mean
income_group,,
Alto,1732,303.33
Bajo,1900,458.44
Medio,3208,363.37
Sin ingresos,2294,597.17


In [16]:
# Cruzamos grupo de ingreso con tipo de póliza
df.groupby(['income_group', 'policy_type'])['total_claim_amount'].agg(['count', 'mean']).round(2)

count    mean
income_group policy_type                  
Alto         Corporate Auto    385  296.69
             Personal Auto    1266  305.70
             Special Auto       81  297.85
Bajo         Corporate Auto    385  442.86
             Personal Auto    1440  462.20
             Special Auto       75  466.38
Medio        Corporate Auto    692  356.49
             Personal Auto    2384  364.21
             Special Auto      132  384.35
Sin ingresos Corporate Auto    503  613.65
             Personal Auto    1700  591.36
             Special Auto       91  614.50

**Conclusión:** Crear categorías a partir de una variable numérica continua
(`income` → `income_group`) facilita el análisis y la interpretación. Podemos
observar si el importe medio de reclamación aumenta, disminuye o se mantiene
estable a medida que el nivel de ingresos cambia, y si ese patrón es consistente
entre los distintos tipos de póliza.

## Challenge 6 — Uso de `get_group` para analizar un segmento concreto

Usamos `get_group()` para extraer un segmento específico y analizarlo en detalle —
igual que en el ejemplo de clase con `grouped.get_group('female')`.

In [17]:
# Agrupamos por tipo de póliza
grouped_policy = df.groupby('policy_type')

# Vemos qué grupos existen y su tamaño
grouped_policy.size()

policy_type
Corporate Auto    1965
Personal Auto     6790
Special Auto       379
dtype: int64

In [18]:
# Extraemos el grupo 'Personal Auto' para analizarlo en detalle
personal_auto = grouped_policy.get_group('Personal Auto')

print(f'Número de clientes con Personal Auto: {len(personal_auto)}')
personal_auto[['income', 'customer_lifetime_value', 'total_claim_amount']].describe().round(2)

Número de clientes con Personal Auto: 6790


,income,customer_lifetime_value,total_claim_amount
count,6790.00,6784.00,6790.00
mean,37724.14,185259.15,430.95
std,30308.27,450238.32,286.39
min,0.00,1898.01,0.38
25%,0.00,4626.80,272.64
50%,33844.50,7664.05,376.52
75%,62436.00,26197.41,547.20
max,99961.00,5816655.35,2893.24


In [19]:
# Dentro de este segmento, vemos el importe medio de reclamación por clase de vehículo
personal_auto.groupby('vehicle_class')['total_claim_amount'].agg(['count', 'mean']).round(2).sort_values('mean', ascending=False)

,count,mean
vehicle_class,,
Luxury SUV,144,1087.13
Luxury Car,114,1079.24
SUV,1316,574.43
Sports Car,354,549.15
Four-Door Car,3444,349.76
Two-Door Car,1418,346.72


**Conclusión:** Aislar un segmento concreto con `get_group()` permite hacer un análisis
más profundo sin perder de vista el contexto general. Dentro de Personal Auto,
podemos identificar qué clases de vehículo generan reclamaciones más costosas
en promedio — información útil para ajustar primas por clase de vehículo.

## Resumen final

A lo largo de este laboratorio hemos aplicado:

- **Filtrado por percentiles** para identificar clientes de riesgo (Challenge 1)
- **`groupby` + `unstack`** para tablas legibles por dos variables (Challenge 2)
- **Filtrado previo a la agregación** para analizar subgrupos específicos (Challenge 3)
- **`agg()` con múltiples métricas** y filtrado posterior por tamaño de grupo (Challenge 4)
- **`apply()` con función personalizada** para crear categorías a partir de variables numéricas (Challenge 5)
- **`get_group()`** para profundizar en un segmento concreto (Challenge 6)

Estas técnicas combinadas de agregación y filtrado permiten pasar de un dataset
general a información segmentada y accionable para el negocio.